# Telugu / Code-mixed Abusive Content Detection — Full Experimental Pipeline

Runs on Colab (GPU runtime: Runtime > Change runtime type > T4/A100).

Covers: baselines (mBERT, IndicBERT, MuRIL, XLM-R full fine-tuning), PEFT sweep (LoRA, QLoRA, AdaLoRA, DoRA, IA3), multi-seed significance study, ablations (rank, quantization, target modules), cross-domain transfer (telugu <-> codemixed), and paper-ready result tables.

**Upload your `telugu_*.csv` and `codemixed_*.csv` files into the `data/` folder before running**, or mount Google Drive and copy them in.

In [ ]:
!git clone <YOUR_REPO_URL_OR_UPLOAD_MANUALLY> repo 2>/dev/null || echo 'Upload the project folder manually instead if not using git.'
%cd repo
!pip install -q -r requirements.txt

In [ ]:
# Verify data is present
import os
for f in ['telugu_train.csv','telugu_val.csv','telugu_test.csv',
          'codemixed_train.csv','codemixed_validation.csv','codemixed_test.csv']:
    path = os.path.join('data', f)
    print(f, 'OK' if os.path.exists(path) else 'MISSING')

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## Stage 1 — Baseline full fine-tuning (mBERT, IndicBERT, MuRIL, XLM-R)

In [ ]:
!python src/run_experiments.py --stage baselines --domain telugu --epochs 5

## Stage 2 — PEFT comparison on the strongest backbone (LoRA, QLoRA, AdaLoRA, DoRA, IA3, + Full-FT on same backbone)

In [ ]:
!python src/run_experiments.py --stage peft --domain telugu --epochs 5

## Stage 3 — Multi-seed statistical significance study (5 seeds x {full, lora, qlora})

In [ ]:
!python src/run_experiments.py --stage seeds --domain telugu --epochs 5

## Stage 4 — Ablations: LoRA rank, quantization, target modules

In [ ]:
!python src/run_experiments.py --stage ablation_rank --domain telugu --epochs 3
!python src/run_experiments.py --stage ablation_quant --domain telugu --epochs 3
!python src/run_experiments.py --stage ablation_targets --domain telugu --epochs 3

## Stage 5 — Cross-domain transfer (pure Telugu <-> code-mixed Telugu-English)

In [ ]:
!python src/run_experiments.py --stage cross_domain --epochs 5

## Stage 6 — Aggregate results + paper tables

In [ ]:
!python src/run_experiments.py --stage aggregate
!python src/analysis.py --action main_table --domain telugu
!python src/analysis.py --action seed_summary
!python src/analysis.py --action significance --method_a full --method_b qlora
!python src/analysis.py --action significance --method_a lora --method_b qlora

## Stage 7 — Error analysis on the best model
Replace `MODEL_DIR` with whichever run had the highest macro-F1 in the aggregated table.

In [ ]:
MODEL_DIR = 'results/peft__qlora__telugu/model'  # update after checking results/all_results.csv
!python src/analysis.py --action error_analysis --model_dir {MODEL_DIR} --domain telugu

## Stage 8 — Quick inference sanity check on the best model

In [ ]:
!python src/infer.py --model_dir {MODEL_DIR} --base_model_for_adapter xlm-roberta-base --text "<paste a Telugu test sentence here>"

## Stage 9 (optional) — Export to ONNX for fast production inference

In [ ]:
!python src/export_onnx.py --model_dir {MODEL_DIR} --base_model_for_adapter xlm-roberta-base --out_dir onnx_model --quantize

## Download all results (metrics, tables, confusion matrices, error analysis) as a zip

In [ ]:
!zip -r results_bundle.zip results/
from google.colab import files
files.download('results_bundle.zip')